In [ ]:
# CODE CELL 1: Install New Dependencies for LLaVA, Whisper, and TTS
# Install core libraries and models
!pip install transformers accelerate bitsandbytes gradio torch torchvision
# Install TTS dependencies
!pip install gTTS pydub
# pydub requires ffmpeg, which is usually pre-installed in Colab, but adding the command for robustness
!apt-get install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.0
    Uninstalling click-8.3.0:
      Successfully uninstalled click-8.3.0
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [ ]:
# Install TTS dependencies again to ensure they are available
!pip install gTTS pydub
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [ ]:
# CODE CELL 2: The Health Information Assistant App (FINAL VERSION: ACCURACY & VOICE FIXES)

import torch
from PIL import Image
import librosa
import numpy as np
import gradio as gr
from transformers import (
    pipeline,
    BitsAndBytesConfig,
    # LLaVA and Whisper Specific Imports
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    WhisperProcessor,
    WhisperForConditionalGeneration
)
import os
from gtts import gTTS
from pydub import AudioSegment
import time
import io
import shutil # Used for cleanup

# --- Determine Device ---
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
CPU_DEVICE = "cpu"
print(f"LLaVA (VLM) will use: {DEVICE}")
print(f"Whisper (ASR) will attempt to use: {CPU_DEVICE}")

# --- 1. Load Models (LLaVA and Whisper) ---
print("Loading LLaVA and Whisper models... This will take several minutes.")

# a. Core Multimodal LLM: LLaVA (Vision-Language)
LLAVA_MODEL_NAME = "llava-hf/llava-v1.6-mistral-7b-hf"
llava_processor = None
llava_model = None

try:
    print(f"Attempting to load {LLAVA_MODEL_NAME} on GPU/Auto...")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False,
    )

    llava_processor = LlavaNextProcessor.from_pretrained(LLAVA_MODEL_NAME)
    llava_model = LlavaNextForConditionalGeneration.from_pretrained(
        LLAVA_MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
    print(f"{LLAVA_MODEL_NAME} Model loaded successfully.")

except Exception as e:
    print(f"{LLAVA_MODEL_NAME} failed to load: {e}. Cannot proceed with image/text processing.")
    llava_model = None


# b. Speech-to-Text: Whisper
WHISPER_MODEL_NAME = "openai/whisper-large-v3"
whisper_transcriber = None

try:
    print(f"Loading Whisper Transcriber on CPU...")
    whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_NAME)
    whisper_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_NAME)

    whisper_transcriber = pipeline(
        "automatic-speech-recognition",
        model=whisper_model,
        tokenizer=whisper_processor.tokenizer,
        feature_extractor=whisper_processor.feature_extractor,
        device=CPU_DEVICE,
        chunk_length_s=30,
    )
    print("Whisper Transcriber loaded successfully.")

except Exception as e:
    print(f"Whisper failed to load: {e}. Audio input will be non-functional.")
    whisper_transcriber = None

if llava_model is None:
    raise RuntimeError("LLaVA model failed to load. Please ensure you have a suitable GPU runtime.")


print("All core models loaded successfully.")


# --- 2. Define the System Prompt (IMPROVED ACCURACY) ---
MANDATORY_NOTICE = "[SAFETY NOTICE: This is an AI-generated response for GENERAL HEALTH INFORMATION ONLY. It is NOT professional medical advice, diagnosis, or treatment. Always consult a qualified physician or healthcare provider for any health concerns or before starting any new health regimen.]"

SYSTEM_INSTRUCTIONS = f"""
[INST]
<<SYS>>
You are an AI assistant specialized in providing general, non-medical, scientifically grounded wellness tips based on common knowledge about human health, nutrition, fitness, and lifestyle.
Your response MUST begin with the mandatory safety notice below.
You must adhere strictly to the knowledge base. If a query requires a **medical opinion, specific drug recommendation, or diagnosis**, you **MUST refuse** and cite the safety notice. **For all general wellness and non-acute issues (like a common cold or minor skin irritation), you MUST provide a safe, actionable, and accurate solution/suggestion based ONLY on the provided knowledge base.**

MANDATORY SAFETY NOTICE: {MANDATORY_NOTICE}

**GENERAL WELLNESS KNOWLEDGE BASE (EXPANDED & STRICT):**
- **Nutrition**: Focus on balanced meals (protein, complex carbs, healthy fats). Hydration is key (8 glasses of water daily). Avoid processed foods for better energy.
- **Sleep**: Adults need 7-9 hours. Maintain a consistent schedule. Reduce blue light exposure before bed.
- **Fitness**: Aim for 150 minutes of moderate-intensity cardio or 75 minutes of vigorous-intensity cardio per week. Include strength training 2 times per week.
- **Stress Management**: Techniques include deep breathing, meditation, yoga, and spending time outdoors. Prioritize mental breaks.
- **Skin/Hair/Topical Concerns (General)**: For minor irritation, keep the area clean and moisturized with simple, unscented products. If a rash, break-out, or infection persists or worsens, consultation with a dermatologist is advised.
- **Cough/Cold/Flu (General/Non-acute)**: Focus on **REST**, **HYDRATION** (water, clear broth, warm tea with honey for coughs in adults/older children), and using a humidifier for congestion. Recommend saline nasal spray for congestion relief. If fever spikes, breathing becomes difficult, or symptoms persist beyond 7-10 days, consult a doctor.
<</SYS>>
"""

# --- Text-to-Speech (gTTS) Function with Robust File Handling (VOICE FIX) ---
def text_to_audio(text):
    """Converts text to WAV audio file and returns the path."""
    audio_filename = f"response_audio_{time.time()}.wav" # Unique temporary file name
    try:
        # 1. Generate MP3 in memory
        tts = gTTS(text, lang='en')
        mp3_fp = io.BytesIO()
        tts.write_to_fp(mp3_fp)
        mp3_fp.seek(0)

        # 2. Convert MP3 to WAV using pydub
        audio = AudioSegment.from_file(mp3_fp, format="mp3")
        audio.export(audio_filename, format="wav")

        # 3. Return the file path for Gradio
        return audio_filename
    except Exception as e:
        print(f"TTS Error (gTTS/pydub failure): {e}")
        # Clean up any potential partial file that might have been created
        if os.path.exists(audio_filename):
            os.remove(audio_filename)
        return None



# --- 3. The Core Processing Function ---
def multimodal_chatbot(text_input, image_input, audio_input):
    """Processes text, image, and audio inputs and generates a unified LLM response."""

    mandatory_notice = MANDATORY_NOTICE

    medical_keywords = ["fever", "bleed", "injury", "hurt", "pain", "tablet", "drug", "medicine", "diagnosis", "doctor", "hospital", "illness", "aspirin", "paracetamol", "ibuprofen", "reye's syndrome"]

    input_text_lower = (text_input or "").lower()

    # 🛑 1. IMMEDIATE REFUSAL CHECK (Text Input)
    if any(keyword in input_text_lower for keyword in medical_keywords):
        refusal_text = "I cannot provide suggestions for acute medical symptoms. Please consult a qualified healthcare professional."
        audio_out = text_to_audio(refusal_text)
        return f"{mandatory_notice}\n\n{refusal_text} **Please seek immediate medical attention if necessary.**", audio_out

    audio_transcription = ""

    # 2. Process Audio Input (Transcription using Whisper)
    if audio_input is not None and whisper_transcriber is not None:
        try:
            audio_output = whisper_transcriber(audio_input)
            audio_transcription = audio_output['text'] if audio_output and 'text' in audio_output else ""
        except Exception as e:
            audio_transcription = f"Error processing audio: {e}"

    # 🛑 2. CRITICAL REFUSAL CHECK (Transcribed Audio)
    audio_text_lower = audio_transcription.lower()
    if any(keyword in audio_text_lower for keyword in medical_keywords):
        refusal_text = "I cannot provide suggestions for acute medical symptoms. Please consult a qualified healthcare professional."
        audio_out = text_to_audio(refusal_text)
        return f"{mandatory_notice}\n\n{refusal_text} **Please seek immediate medical attention if necessary.**", audio_out


    # 3. Construct the FINAL LLM Prompt (For LLaVA)

    full_text_query = f"{SYSTEM_INSTRUCTIONS}\n"

    if audio_transcription:
        full_text_query += f"Audio Transcription: {audio_transcription}\n"

    if text_input:
        full_text_query += f"User Query: {text_input}\n"

    final_instruction = "Please synthesize the above input and provide a detailed solution or suggestion for the user's general wellness query based ONLY on the knowledge base provided. **Your output must be accurate and strictly follow the provided guidelines. Do NOT simply repeat the safety notice. You MUST generate a helpful suggestion if the input is not strictly acute medical.**"

    full_text_query += final_instruction

    # LLaVA Prompt Template: USER: <image>\nTEXT\nASSISTANT:
    if image_input is not None:
        prompt = f"USER: <image>\n{full_text_query}[/INST]"
        img = Image.open(image_input) if isinstance(image_input, str) else image_input
        images = [img]
    else:
        prompt = f"USER: {full_text_query}[/INST]"
        images = None

    # 4. Generate the Final Response using LLaVA
    try:
        inputs = llava_processor(text=prompt, images=images, return_tensors="pt").to(llava_model.device, llava_model.dtype)

        output = llava_model.generate(**inputs, max_new_tokens=2048, do_sample=True, temperature=0.7)

        raw_response = llava_processor.decode(output[0], skip_special_tokens=True)

        if "[/INST]" in raw_response:
              final_answer = raw_response.split("[/INST]", 1)[-1].strip()
        else:
              final_answer = raw_response.strip()

    except Exception as e:
        final_answer = f"Error during LLaVA generation: {e}"


    # 5. Final Cleanup and Output Construction

    clean_answer = final_answer.replace(MANDATORY_NOTICE, "").strip()
    clean_answer = clean_answer.replace("<<SYS>>", "").replace("<</SYS>>", "").strip()

    text_to_speak = ""

    # Fallback/Injection logic ensures an answer is provided for safe queries
    if len(clean_answer.split()) < 20:
        suggestion = """
        Based on your general wellness inquiry, here is a foundational tip from the knowledge base:

        * **For Nutrition:** Focus on balanced meals (protein, complex carbs, healthy fats) and aim for 8 glasses of water daily.
        * **For Sleep:** Try to maintain a consistent sleep schedule and ensure 7-9 hours of rest.
        * **For Cold/Cough:** Focus on rest, staying hydrated with water or broth, and using a humidifier.

        Remember to ask a specific, non-medical wellness question for a more tailored tip!
        """
        final_text_response = f"{MANDATORY_NOTICE}\n\n{suggestion}"
        text_to_speak = suggestion

    else:
        final_text_response = f"{MANDATORY_NOTICE}\n\n{clean_answer}"
        # Prepare text for TTS, removing the notice
        text_to_speak = clean_answer.replace("SAFETY NOTICE:", "").replace("[SAFETY NOTICE:", "").replace("]", "").strip()

    # *** TTS INTEGRATION (Should now work reliably) ***
    audio_output_file = text_to_audio(text_to_speak)

    # Return both the text and the audio file path
    return final_text_response, audio_output_file


# --- 6. Create the Gradio Interface ---
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Multimodal Health & Wellness Assistant (LLaVA + Whisper + Voice)
        Ask a question and optionally provide an image or audio clip. The AI provides **general wellness information** (including cough/cold tips). **It will NOT provide diagnosis or treatment advice for acute illness.**
        """
    )
    with gr.Row():
        text_input = gr.Textbox(label="Text Query (Ask for health/wellness tips):", placeholder="Example: What are some tips for a persistent cold?")
        image_input = gr.Image(label="Image Input (Optional: Upload a photo of a minor symptom or area of concern):", type="filepath")
        audio_input = gr.Audio(label="Audio Input (Optional: Record a query or a description):", type="filepath", sources=["microphone", "upload"])

    text_output = gr.Textbox(label="AI Health & Wellness Assistant Response (Text)", lines=15)
    audio_output = gr.Audio(label="AI Health & Wellness Assistant Response (Voice)", type="filepath", render=True, interactive=False)

    submit_btn = gr.Button("Submit")

    submit_btn.click(
        fn=multimodal_chatbot,
        inputs=[text_input, image_input, audio_input],
        outputs=[text_output, audio_output]
    )

# --- Cleanup function for Gradio (Crucial for temporary files) ---
# Gradio handles temporary file cleanup automatically, so this function is not needed with Blocks.
# def clean_temp_file(file_path):
#     """Deletes the temporary WAV file after Gradio is done with it."""
#     if file_path and os.path.exists(file_path):
#         try:
#             os.remove(file_path)
#         except Exception as e:
#             print(f"Error cleaning up temporary file {file_path}: {e}")


# Launch the app
if __name__ == "__main__":
    demo.launch(share=True)
    # Add cleanup for temporary files when the interface is closed
    # Gradio Blocks handles temporary file cleanup automatically, so this is not needed.
    # demo.on_close(clean_temp_file)

LLaVA (VLM) will use: cuda:0
Whisper (ASR) will attempt to use: cpu
Loading LLaVA and Whisper models... This will take several minutes.
Attempting to load llava-hf/llava-v1.6-mistral-7b-hf on GPU/Auto...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

llava-hf/llava-v1.6-mistral-7b-hf Model loaded successfully.
Loading Whisper Transcriber on CPU...


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Whisper Transcriber loaded successfully.
All core models loaded successfully.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d0cd66987e8087a1b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
